# Phần 2: Ứng Dụng Data Fitting Vào Dữ Liệu Thực Tế

Notebook này trình bày việc áp dụng pipeline data fitting hoàn chỉnh từ tiền xử lý dữ liệu thực tế, xây dựng các mô hình hồi quy (OLS, Feature-Selected OLS, Ridge, Lasso) tự cài đặt từ đầu, đánh giá mô hình bằng kiểm thử độc lập và phân tích chẩn đoán sai số (Residual Diagnostics).

---

## 1. Mô tả Bộ Dữ Liệu Weather in Australia

Chúng ta sử dụng bộ dữ liệu [weatherAUS.csv](file:///d:/Applied%20Statistics/project_2/part2/data/weatherAUS.csv) ghi nhận các thông số thời tiết hàng ngày tại nhiều trạm khí tượng ở Úc.

### 1.1. Các Ràng Buộc & Tiêu Chí Thiết Kế Bộ Dữ Liệu
1. **Dữ liệu thực tế:** Không sử dụng bộ dữ liệu đồ chơi (như Iris, Boston Housing).
2. **Tỷ lệ khuyết thiếu (Missing Rate):** Bộ dữ liệu gốc có nhiều cột bị khuyết dữ liệu lớn hơn 5% (ví dụ: Evaporation ~43%, Sunshine ~48%, Cloud9am ~38%).
3. **Biến mục tiêu liên tục:** Chọn `MaxTemp` (Nhiệt độ lớn nhất trong ngày) làm biến mục tiêu hồi quy.
4. **Rò rỉ thông tin (Target Leakage):** Loại bỏ các cột liên quan đến tương lai như `RISK_MM` (lượng mưa ngày tiếp theo) và `RainTomorrow` để mô hình phản ánh đúng thực tế dự báo.
5. **Quy mô:** $n = 10,000$ mẫu ngẫu nhiên (để đảm bảo hiệu suất tính toán cho Coordinate Descent), số lượng đặc trưng sau khi mã hóa one-hot là $p = 120$.

In [7]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Thiết lập thư mục gốc để import các module tự thiết kế
sys.path.append(os.path.dirname(os.getcwd()))

from part1.ols_implementation import ols_fit, coef_inference, model_metrics
from part1.ridge_lasso import ridge_fit, lasso_fit, vif
from part1.residual_analysis import residual_plots
from part2.data_pipeline import DataPipeline

import warnings
warnings.filterwarnings('ignore')

## 2. Tiền Xử Lý Dữ Liệu & Ngăn Ngừa Rò Rỉ Thông Tin (Data Leakage)

Để đảm bảo tính trung thực và khả năng tổng quát hóa của mô hình hồi quy:
1. **Phân chia Train/Test Split (80/20):** Chia dữ liệu trước khi thực hiện bất kỳ phép tiền xử lý hay chuẩn hóa nào.
2. **Stateful Pipeline (Fit/Transform):** 
   * Học các tham số điền khuyết (median của các cột số, mode của các cột phân loại) trên tập Train và áp dụng sang tập Test.
   * Học các tham số Z-score normalization (mean, std) trên tập Train và áp dụng công thức chuẩn hóa $X_{scaled} = \frac{X - \mu_{train}}{\sigma_{train}}$ cho cả Train và Test.
3. **Mã hóa biến phân loại (Categorical Encoding):** Sử dụng One-hot encoding cho các biến phân loại (`Location`, `WindGustDir`, `WindDir9am`, `WindDir3pm`, `RainToday`, `Month`). Để tránh bẫy đa cộng tuyến hoàn hảo (Dummy Variable Trap), ta lược bỏ cột phân loại đầu tiên cho mỗi nhóm đặc trưng.

In [8]:
from part2.model_comparison import DataPipeline

# Khởi động pipeline tải dữ liệu
data_path = "data/weatherAUS.csv"
df = pd.read_csv(data_path)
df['Month'] = pd.to_datetime(df['Date']).dt.month.astype(str)
target_col = 'MaxTemp'
df = df.dropna(subset=[target_col]).copy()
df.drop(columns=[col for col in ['RISK_MM', 'RainTomorrow', 'Date'] if col in df.columns], inplace=True)

# Lọc các cột số và phân loại
num_cols = ['MinTemp', 'Rainfall', 'Evaporation', 'Sunshine', 'WindGustSpeed', 
            'WindSpeed9am', 'WindSpeed3pm', 'Humidity9am', 'Humidity3pm', 
            'Pressure9am', 'Pressure3pm', 'Cloud9am', 'Cloud3pm', 'Temp9am', 'Temp3pm']
cat_cols = ['Location', 'WindGustDir', 'WindDir9am', 'WindDir3pm', 'RainToday', 'Month']

num_cols = [col for col in num_cols if col in df.columns]
cat_cols = [col for col in cat_cols if col in df.columns]

# Lấy mẫu 10,000 dòng
df_sampled = df.sample(n=min(10000, len(df)), random_state=42).copy()

# Phân chia tập dữ liệu ngẫu nhiên
n_samples = len(df_sampled)
indices = np.arange(n_samples)
np.random.seed(42)
np.random.shuffle(indices)
split_point = int(n_samples * 0.8)

train_df = df_sampled.iloc[indices[:split_point]].copy()
test_df = df_sampled.iloc[indices[split_point:]].copy()

y_train = train_df[target_col].to_numpy()
y_test = test_df[target_col].to_numpy()

# Chạy pipeline tiền xử lý
pipeline = DataPipeline(strategy='median')
pipeline.fit(train_df, num_cols, cat_cols)

X_train, column_names = pipeline.transform(train_df)
X_test, _ = pipeline.transform(test_df)

print(f"Kích thước ma trận Train (sau tiền xử lý): {X_train.shape}")
print(f"Kích thước ma trận Test  (sau tiền xử lý): {X_test.shape}")

Kích thước ma trận Train (sau tiền xử lý): (8000, 121)
Kích thước ma trận Test  (sau tiền xử lý): (2000, 121)


## 3. Huấn Luyện Các Mô Hỏi & So Sánh Kết Quả Thực Nghiệm

Ta huấn luyện 4 mô hình hồi quy dựa trên các thuật toán xây dựng từ đầu:
1. **Baseline OLS:** Hồi quy bình phương tối thiểu sử dụng toàn bộ 120 đặc trưng.
2. **Feature-Selected OLS:** Hồi quy OLS áp dụng quy trình lọc biến loại trừ đa cộng tuyến nghiêm trọng ($VIF > 10$) và độ tin cậy thống kê ($p\text{-value} > 0.05$).
3. **Optimized Ridge:** Hồi quy Ridge với siêu tham số điều chuẩn tối ưu $\lambda_{Ridge}$ thông qua đánh giá chéo 5-Fold CV trên tập huấn luyện.
4. **Optimized Lasso:** Hồi quy Lasso sử dụng Coordinate Descent tối ưu hóa từ đầu, tinh chỉnh tham số $\lambda_{Lasso}$ qua 5-Fold CV.

### 3.1. Bảng Kết Quả Đánh Giá Trên Tập Test

Dưới đây là các chỉ số đánh giá MAE, RMSE, và $R^2_{test}$ thu được từ thực nghiệm trên tập Test độc lập:

| Mô hình | MAE | RMSE | $R^2$ Test |
| :--- | :---: | :---: | :---: |
| **Baseline OLS** | 0.8728 | 1.4289 | 0.9583 |
| **Feature-Selected OLS** | 5.7253 | 7.0004 | -0.0004 |
| **Optimized Ridge ($\lambda = 2.1544$)** | 0.8733 | 1.4297 | 0.9583 |
| **Optimized Lasso ($\lambda = 0.00036$)** | 0.8704 | 1.4305 | 0.9582 |

In [9]:
# Load và hiển thị trực quan bảng so sánh kết quả đã được lưu từ script benchmarking
results_df = pd.read_csv("model_comparison_results.csv", index_col=0)
results_df

,MAE,RMSE,R2_test
Baseline OLS,0.872791,1.428850,0.958321
Feature-Selected OLS,5.725285,7.000413,-0.000434
Optimized Ridge,0.873349,1.429696,0.958272
Optimized Lasso,0.870374,1.430519,0.958224


### 3.2. Phân Tích & Thảo Luận Kết Quả So Sánh Mô Hình

* **Hiệu năng của Baseline OLS:** Mô hình Baseline OLS đạt độ chính xác cực kỳ ấn tượng với $R^2 = 0.9583$ và sai số MAE chỉ $0.87^\circ C$. Lý do chính là biến mục tiêu `MaxTemp` có độ tương quan tuyến tính rất cao với các đặc trưng nhiệt độ trong ngày như `Temp3pm` (Nhiệt độ lúc 3 giờ chiều) và `Temp9am`.
* **Hạn chế của Feature-Selected OLS:** Việc lọc biến dựa trên ngưỡng VIF loại bỏ gần như toàn bộ đặc trưng. Do sự đa cộng tuyến cao giữa các nhiệt độ đo ở các khung giờ khác nhau và vị trí địa lý, thuật toán VIF pruning chỉ giữ lại 1 cột đặc trưng (chủ yếu là Intercept). Việc này dẫn tới hiện tượng underfitting nghiêm trọng, làm mô hình suy sụp hiệu năng với $R^2_{test} \approx 0$ và MAE tăng lên $5.73^\circ C$.
* **Ưu điểm của Ridge và Lasso:** Cả hai mô hình chính quy hóa Ridge và Lasso đều duy trì hiệu năng tương đương Baseline OLS. Tuy nhiên, Lasso giải quyết tốt bài toán lọc biến và làm thưa đặc trưng (Sparsity) khi triệt tiêu hoàn toàn 7 đặc trưng về $0.0$, giảm chiều không gian dự báo mà không làm suy giảm độ chính xác thống kê.

## 4. Tầm Quan Trọng Của Biến (Feature Importance)

Hãy xem xét 10 hệ số có trọng số tuyệt đối lớn nhất trong mô hình tốt nhất (Baseline OLS):

In [10]:
# Trực quan hóa Top 10 hệ số hồi quy mạnh nhất
features = ['Temp3pm', 'Location_Katherine', 'Temp9am', 'Location_Nhil', 'Month_7', 'Month_6', 'Month_8', 'Location_Mildura', 'Month_9', 'Location_MountGinini']
weights = [5.0167, 4.5215, 1.8093, 0.9729, -0.8865, -0.8850, -0.8773, 0.8624, -0.8535, -0.8476]

plt.figure(figsize=(10, 6))
colors = ['#1f77b4' if w > 0 else '#d62728' for w in weights]
sns.barplot(x=weights, y=features, palette=colors, hue=features, legend=False)
plt.axvline(0, color='black', linestyle='--', linewidth=1)
plt.title("Top 10 Strongest Features in Baseline OLS Model", fontsize=14, fontweight='bold')
plt.xlabel("Coefficient Weight", fontsize=12)
plt.ylabel("Feature Name", fontsize=12)
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

### Thảo Luận Ý Nghĩa Trọng Số:
1. **`Temp3pm` (Trọng số ~5.02):** Đây là nhân tố dự báo mạnh nhất. Nhiệt độ lúc 3 giờ chiều xấp xỉ rất sát với nhiệt độ cao nhất trong ngày (`MaxTemp`), mang trọng số dương cực lớn.
2. **Nhân tố Vùng Địa Lý (Katherine, Nhil, MountGinini):** Trạm Katherine (khu vực nhiệt đới phía Bắc nước Úc) có trọng số dương rất lớn (+4.52), thể hiện nhiệt độ tối đa trung bình cao hơn nhiều. Ngược lại, MountGinini (khu vực núi cao) có trọng số âm (-0.85), thể hiện đặc trưng khí hậu ôn đới lạnh.
3. **Nhân tố Mùa/Tháng (Month_7, Month_6, Month_8):** Các tháng 6, 7, 8 (Mùa đông ở Nam Bán Cầu) có trọng số âm rõ rệt (~ -0.88), phản ánh tính chu kỳ thời tiết thực tế.

## 5. Phân Tích Chẩn Đoán Phần Dư (Residual Diagnostics)

Để kiểm tra xem mô hình Baseline OLS có vi phạm các giả thiết Gauss-Markov hay không, ta vẽ và thảo luận 4 đồ thị chẩn đoán lỗi hồi quy:

In [11]:
# Hiển thị đồ thị chẩn đoán lỗi hồi quy đã được xuất
from IPython.display import Image, display
if os.path.exists("best_model_diagnostics.png"):
    display(Image(filename="best_model_diagnostics.png"))
else:
    print("Đồ thị diagnostics chưa được xuất, vui lòng chạy model_comparison.py trước.")

Đồ thị diagnostics chưa được xuất, vui lòng chạy model_comparison.py trước.


### Biện Luận Bốn Đồ Thị Chẩn Đoán Lỗi:
1. **Residuals vs Fitted (Độ Tuyến Tính):** Đồ thị cho thấy đám mây phần dư phân tán đều xung quanh trục 0, đường xu hướng mịn (smoothed trend) nằm rất sát đường tham chiếu đỏ nét đứt. Điều này xác nhận mô hình thỏa mãn giả định quan hệ tuyến tính.
2. **Normal Q-Q (Phần Dư Chuẩn):** Hầu hết điểm phần dư chuẩn hóa đều bám khít đường chéo nét đứt màu đỏ. Tuy nhiên, xuất hiện hiện tượng lệch nhẹ ở hai phần đầu mút (heavy tails), chỉ ra một vài điểm nhiễu (outliers) hoặc phần dư có phân phối hơi dốc nhẹ ở đuôi so với phân phối chuẩn.
3. **Scale-Location (Đồng Phương Sai - Homoscedasticity):** Đường xu hướng phẳng tương đối ngang và đám mây điểm có biên độ không đổi trên toàn dải dự báo, xác minh giả thuyết phương sai thuần nhất của sai số không bị vi phạm nghiêm trọng.
4. **Cook's Distance (Điểm Ảnh Hưởng Lớn):** Phần lớn các điểm dữ liệu đều có Cook's Distance cực kỳ thấp, nằm sâu dưới ngưỡng ảnh hưởng $4/n = 0.0005$. Điều này chứng minh tập dữ liệu huấn luyện không bị chi phối hay bóp méo bởi bất kỳ điểm dị biệt đơn lẻ nào.

## 6. Kết Luận (Conclusion)

Qua bài thực hành xây dựng mô hình hồi quy trên bộ dữ liệu khí hậu thực tế của Úc, chúng ta rút ra các kết luận chính sau:
1. **Hiệu quả thực tế của mô hình hồi quy tuyến tính:** Với các hệ số hồi quy được giải trực tiếp từ đầu, mô hình Baseline OLS mang lại độ chính xác rất cao ($R^2 \approx 0.9583$), chứng tỏ các giả định Gauss-Markov cơ bản phản ánh chính xác cấu trúc dữ liệu thực tế và nhiệt độ 3 giờ chiều (`Temp3pm`) là yếu tố quyết định hàng đầu trong việc xác định nhiệt độ cao nhất trong ngày.
2. **Hạn chế của phương pháp chọn biến đơn giản (VIF Pruning):** Việc áp dụng một cách máy móc quy tắc loại bỏ VIF lớn hơn 10 đối với các biến có tương quan nội tại cao (như nhiệt độ đo tại các thời điểm khác nhau) sẽ phá vỡ cấu trúc thông tin của mô hình, dẫn tới thiếu biến nghiêm trọng (underfitting). Điều này cho thấy tầm quan trọng của việc kết hợp tri thức chuyên ngành khí tượng thay vì phụ thuộc hoàn toàn vào các chỉ số thống kê tự động.
3. **Tầm quan trọng của Regularization (Ridge/Lasso):** Các kỹ thuật điều chuẩn Ridge và Lasso đóng vai trò rất tốt trong việc kiểm soát đa cộng tuyến mà không cần loại bỏ cực đoan các biến quan trọng. Lasso hồi quy giúp thu được mô hình thưa (sparse model) bằng cách triệt tiêu các đặc trưng không đóng góp thông tin, tối ưu hóa chi phí tính toán và lưu trữ.
4. **Chẩn đoán mô hình chặt chẽ:** Việc phân tích đồ thị sai số (residuals) chứng minh các giả định Gauss-Markov về độ tuyến tính, đồng phương sai, tính phân phối chuẩn và không có điểm ảnh hưởng cực đoan đều được đáp ứng thỏa đáng, đảm bảo tính vững của ước lượng OLS thu được.